## 1. Завантаження та очищення даних

In [2]:
import pandas as pd
import numpy as np
import timeit

# Завантажуємо дані. Враховуємо, що розділювач — крапка з комою.
# '?' — це пропуски в цьому датасеті.
df = pd.read_csv('household_power_consumption.txt', sep=';', 
                 na_values='?', low_memory=False)

# Видаляємо рядки з пропущеними значеннями
df = df.dropna()

# Перетворюємо типи в числові (крім дати та часу)
float_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
              'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df[float_cols] = df[float_cols].astype(float)

# Створюємо колонку DateTime для зручної фільтрації за часом
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)

print("Дані завантажено. Розмірність:", df.shape)
df.head()

Дані завантажено. Розмірність: (2049280, 10)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


## 2. Формування вибірок
### 2.1. Записи, у яких загальна активна потужність > 5 кВт

In [3]:
def task_1(df):
    return df[df['Global_active_power'] > 5]

# Заміряємо час
execution_time = timeit.timeit(lambda: task_1(df), number=1)
print(f"Час виконання: {execution_time:.5f} секунд")
display(task_1(df).head())

Час виконання: 0.00502 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


### 2.2. Сила струму 19-20 А, де (стіралка + холодильник) > (бойлер + кондиціонер)

In [4]:
def task_2(df):
    return df[(df['Global_intensity'] >= 19) & 
              (df['Global_intensity'] <= 20) & 
              (df['Sub_metering_1'] + df['Sub_metering_2'] > df['Sub_metering_3'])]

execution_time = timeit.timeit(lambda: task_2(df), number=1)
print(f"Час виконання: {execution_time:.5f} секунд")
display(task_2(df).head())

Час виконання: 0.01714 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


### 2.3. Випадкові 500,000 записів та середні величини груп споживання

In [5]:
def task_3(df):
    sample_df = df.sample(n=500000, replace=False)
    averages = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return averages

execution_time = timeit.timeit(lambda: task_3(df), number=1)
print(f"Час виконання: {execution_time:.5f} секунд")
print("Середні значення:")
print(task_3(df))

Час виконання: 0.11511 секунд
Середні значення:
Sub_metering_1    1.125562
Sub_metering_2    1.292890
Sub_metering_3    6.465054
dtype: float64


### 2.5. Пронормувати та стандартизувати вибраний датасет

In [6]:
def normalize_data(df):
    # Вибираємо колонку для прикладу
    col = df['Global_active_power']
    normalized = (col - col.min()) / (col.max() - col.min())
    
    # Стандартизація (Z-score)
    standardized = (col - col.mean()) / col.std()
    
    return normalized, standardized

norm, std = normalize_data(df)
print("Дані пронормовано та стандартизовано.")

Дані пронормовано та стандартизовано.


### 2.6. Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів

In [7]:
# Коефіцієнти Пірсона та Спірмена
pearson = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Коефіцієнт Пірсона: {pearson:.4f}")
print(f"Коефіцієнт Спірмена: {spearman:.4f}")

Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


### 2.7. Провести One Hot Encoding категоріального атрибута


In [8]:
def apply_one_hot_encoding(df):
    # Беремо невеликий зріз, щоб не забити пам'ять тисячами колонок
    sample_for_ohe = df.head(10).copy()
    
    # Використовуємо pd.get_dummies для колонки 'Date'
    ohe_df = pd.get_dummies(sample_for_ohe['Date'], prefix='Date')
    return ohe_df

print("Результат One Hot Encoding (приклад для перших 10 записів):")
display(apply_one_hot_encoding(df))

Результат One Hot Encoding (приклад для перших 10 записів):


,Date_16/12/2006
0,True
1,True
2,True
3,True
4,True
5,True
6,True
7,True
8,True
9,True
